# <Person letter>: <topic>

Copy this notebook to start a new one. Run the setup cell first.

- Colab: add `HF_TOKEN` under Secrets (key icon). Kaggle: Add-ons > Secrets.
- If Colab asks to restart after the install, restart and run the setup cell again.
- Constants (labels, seeds, model ids, thresholds) come from `config`; do not redefine them here.
- Save checkpoints and results under `ARTIFACTS`, not in the repo.


In [ ]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

%load_ext autoreload
%autoreload 2

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

## Interface check
Use the shared functions (real or fake). Swap in your own when ready.

In [ ]:
from src.detector import detect, explain
from src.rag import retrieve
from src.rewriter import rewrite
from src.reviewers import review

text = "[example chatbot answer]"
print(contract.validate_detect(detect(text)))
print(contract.validate_retrieve(retrieve(text), config.RAG_K)[0][:80])

## Experiments

## Save results
`utils.save_result` stores metrics with the seed, git commit and package versions.

In [ ]:
# utils.save_result("my_experiment", {"metric": 0.0}, extra={"seed": config.SEED})